## 🧠 Skip-Gram Model Architecture and Forward Pass

The Skip-Gram model is a shallow neural network designed to learn word embeddings by predicting the surrounding context words for a given center word. It has three main components:

- **Input Layer**: A one-hot encoded vector representing the center word.
- **Hidden Layer**: A dense vector (the word embedding).
- **Output Layer**: Predicts context words using softmax.

---

### 🏗️ Matrix Dimensions

Let:

- $V$: Vocabulary size
- $D$: Embedding dimension
- $x \in \mathbb{R}^V$: One-hot input vector for the center word
- $W_1 \in \mathbb{R}^{V \times D}$: Input-to-hidden weight matrix
- $W_2 \in \mathbb{R}^{D \times V}$: Hidden-to-output weight matrix

---

### 📤 Forward Pass: Step by Step

#### 1. Hidden Layer (Embedding Lookup)

We project the one-hot vector $x$ into the embedding space using $W_1$:

$$
h = W_1^T x
$$

- $h \in \mathbb{R}^D$ is the embedding of the center word.
- Since $x$ is one-hot, this effectively selects a row of $W_1$.

#### 2. Output Layer (Unnormalized Scores)

We compute the raw scores $u$ using $W_2$:

$$
u = W_2^T h
$$

- $u \in \mathbb{R}^V$ is a vector of scores for all possible output (context) words.

#### 3. Softmax Function (Context Probabilities)

To get a probability distribution over the vocabulary:

$$
\hat{y}_i = \frac{\exp(u_i)}{\sum_{j=1}^{V} \exp(u_j)}
$$

- $\hat{y} \in \mathbb{R}^V$ gives the predicted probabilities for all context words.

---

### 🔁 Summary of the Flow

Given a one-hot vector $x$:
1. Project to hidden layer: $h = W_1^T x$
2. Compute output logits: $u = W_2^T h$
3. Get predictions: $\hat{y} = \text{softmax}(u)$

---

### 🧠 Interpretation

- The hidden layer output $h$ **is the word embedding** for the center word.
- The output $\hat{y}$ represents the model’s belief about which words are likely to appear in the context.
- This model is trained by comparing $\hat{y}$ to the actual context words using **cross-entropy loss**.

---


## 🔄 Backpropagation and Loss in Skip-Gram

Once we compute the predicted context probabilities $\hat{y}$ from the forward pass, we use **backpropagation** to update the model's weights and learn better word embeddings.

---

### 🎯 Objective: Cross-Entropy Loss

For a center word $w_c$ and a true context word $w_o$, we define the target vector $y$ as a one-hot vector where only the index of $w_o$ is 1.

The model’s predicted probability vector is $\hat{y}$.

The loss for a single training pair is:

$$
\mathcal{L} = -\sum_{i=1}^{V} y_i \log(\hat{y}_i) = -\log(\hat{y}_{w_o})
$$

This penalizes the model when it assigns low probability to the actual context word.

---

### 🔁 Backpropagation: Updating Weights

Let’s break down how gradients flow through the model and update weights $W_1$ and $W_2$.

#### 1. Error Term

We compute the error between the predicted distribution $\hat{y}$ and the true distribution $y$:

$$
e = \hat{y} - y
$$

This vector $e \in \mathbb{R}^V$ tells us how much each predicted word probability deviates from the true target.

---

#### 2. Gradient for $W_2$

We update $W_2$ using the outer product of the hidden layer $h$ and the error $e$:

$$
\frac{\partial \mathcal{L}}{\partial W_2} = h \cdot e^T
$$

Then update the weights:

$$
W_2 := W_2 - \eta \cdot \frac{\partial \mathcal{L}}{\partial W_2}
$$

Where $\eta$ is the learning rate.

---

#### 3. Gradient for $W_1$

We propagate the error backward through $W_2$ to update $W_1$:

$$
\frac{\partial \mathcal{L}}{\partial W_1} = x \cdot (W_2 \cdot e)^T
$$

Then update:

$$
W_1 := W_1 - \eta \cdot \frac{\partial \mathcal{L}}{\partial W_1}
$$

Here:
- $x$ is the one-hot vector of the center word
- $W_2 \cdot e$ computes the error in the embedding space

---

### 📝 Summary of Training Step

For each $(\text{center}, \text{context})$ pair:
1. Encode the center word as one-hot vector $x$
2. Compute hidden layer $h = W_1^T x$
3. Compute output logits $u = W_2^T h$
4. Apply softmax: $\hat{y} = \text{softmax}(u)$
5. Compute loss: $\mathcal{L} = -\log(\hat{y}_{\text{context}})$
6. Backpropagate to compute gradients for $W_1$ and $W_2$
7. Update weights with SGD

---

### ⚙️ Intuition

- $W_1$ learns how to **encode** words into vector space (word embeddings)
- $W_2$ learns how to **decode** embeddings into context predictions
- Training adjusts both so that similar words end up with similar embeddings based on their usage

---

### 💡 Optimization Note

The full softmax and gradient update scale with vocabulary size $V$, which can be slow. To fix this in large-scale applications, we can use:
- **Negative Sampling**
- **Hierarchical Softmax**

But in this basic version, we use full softmax and exact gradients.

---


In [1]:
import numpy as np
import re
from collections import defaultdict

# ----------------------------
# 1. Text Preprocessing
# ----------------------------

def tokenize_corpus(corpus):
    """
    Tokenizes input text: lowers case, removes punctuation, and splits by spaces.

    Example:
    >>> tokenize_corpus("The cat sat!")
    ['the', 'cat', 'sat']
    """
    corpus = corpus.lower()
    corpus = re.sub(r'[^\w\s]', '', corpus)  # Remove punctuation
    return corpus.split()

def build_vocab(tokenized_corpus):
    """
    Builds vocabulary and returns word2idx and idx2word dictionaries.

    Example:
    >>> build_vocab(['the', 'cat', 'sat'])
    ({'cat': 0, 'sat': 1, 'the': 2}, {0: 'cat', 1: 'sat', 2: 'the'})
    """
    vocab = sorted(set(tokenized_corpus))
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    return word2idx, idx2word

def generate_training_data(tokenized_corpus, window_size=2):
    """
    Generates (center, context) word index pairs for Skip-Gram training.

    Example:
    For the token list ['the', 'cat', 'sat'], and window_size=1,
    Pairs would be [('the','cat'), ('cat','the'), ('cat','sat'), ('sat','cat')]

    Output: List of (center_idx, context_idx), word2idx
    """
    word2idx, _ = build_vocab(tokenized_corpus)
    pairs = []
    for idx, center_word in enumerate(tokenized_corpus):
        context_range = range(max(0, idx - window_size), min(len(tokenized_corpus), idx + window_size + 1))
        for context_idx in context_range:
            if context_idx != idx:
                pairs.append((word2idx[center_word], word2idx[tokenized_corpus[context_idx]]))
    return pairs, word2idx

# ----------------------------
# 2. Skip-Gram Model Definition
# ----------------------------

class SkipGramModel:
    def __init__(self, vocab_size, embedding_dim=10, learning_rate=0.01):
        """
        Initializes the model with random weights.
        """
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.lr = learning_rate

        # Input embedding matrix (vocab_size x embedding_dim)
        self.W1 = np.random.randn(vocab_size, embedding_dim) * 0.01

        # Output weight matrix (embedding_dim x vocab_size)
        self.W2 = np.random.randn(embedding_dim, vocab_size) * 0.01

    def softmax(self, x):
        """
        Computes softmax vector from raw scores.
        """
        e_x = np.exp(x - np.max(x))
        return e_x / np.sum(e_x)

    def forward(self, x):
        """
        Forward pass through the network.
        x is a one-hot encoded word vector (vocab_size,)
        Returns: predicted probability vector, hidden layer
        """
        h = np.dot(self.W1.T, x)              # Hidden layer (embedding)
        u = np.dot(self.W2.T, h)              # Raw scores for output words
        y_pred = self.softmax(u)              # Probabilities
        return y_pred, h

    def backprop(self, x, h, y_pred, y_true):
        """
        Backpropagation step to compute gradients and update weights.
        """
        e = y_pred - y_true                    # Error vector
        dW2 = np.outer(h, e)                   # Gradient for W2
        dW1 = np.outer(x, np.dot(self.W2, e))  # Gradient for W1

        # Update weights
        self.W1 -= self.lr * dW1
        self.W2 -= self.lr * dW2

    def train(self, training_data, epochs=1000, print_loss=False):
        """
        Trains the model using the training data for a given number of epochs.
        """
        for epoch in range(epochs):
            loss = 0
            for center_idx, context_idx in training_data:
                # One-hot encode the input and target
                x = np.zeros(self.vocab_size)
                x[center_idx] = 1
                y_true = np.zeros(self.vocab_size)
                y_true[context_idx] = 1

                # Forward pass
                y_pred, h = self.forward(x)

                # Compute loss and backprop
                loss -= np.log(y_pred[context_idx] + 1e-9)
                self.backprop(x, h, y_pred, y_true)

            if print_loss and epoch % 100 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def get_embedding(self):
        """
        Returns the learned word embedding matrix.
        Each column is the vector for a word.
        """
        return self.W1.T

# ----------------------------
# 3. Example Usage
# ----------------------------

if __name__ == "__main__":
    # Example sentence
    corpus = "the quick brown fox jumps over the lazy dog"

    # Tokenize and generate training data
    tokens = tokenize_corpus(corpus)
    print("Tokens:", tokens)

    training_pairs, word2idx = generate_training_data(tokens, window_size=2)
    print("Training Pairs (index):", training_pairs)

    # Create model
    vocab_size = len(word2idx)
    model = SkipGramModel(vocab_size=vocab_size, embedding_dim=10, learning_rate=0.01)

    # Train model
    model.train(training_pairs, epochs=1000, print_loss=True)

    # Get final embeddings
    embeddings = model.get_embedding()

    # Print embedding for a word
    word = "fox"
    print(f"\nEmbedding for '{word}':")
    print(embeddings[:, word2idx[word]])


Tokens: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
Training Pairs (index): [(7, 6), (7, 0), (6, 7), (6, 0), (6, 2), (0, 7), (0, 6), (0, 2), (0, 3), (2, 6), (2, 0), (2, 3), (2, 5), (3, 0), (3, 2), (3, 5), (3, 7), (5, 2), (5, 3), (5, 7), (5, 4), (7, 3), (7, 5), (7, 4), (7, 1), (4, 5), (4, 7), (4, 1), (1, 7), (1, 4)]
Epoch 0, Loss: 62.3819
Epoch 100, Loss: 62.1474
Epoch 200, Loss: 55.9137
Epoch 300, Loss: 46.5289
Epoch 400, Loss: 43.7207
Epoch 500, Loss: 42.9362
Epoch 600, Loss: 42.6205
Epoch 700, Loss: 42.4881
Epoch 800, Loss: 42.4359
Epoch 900, Loss: 42.4156

Embedding for 'fox':
[-0.69247263  0.4236162   0.49291615 -0.70842537  0.49790544  1.40208645
 -0.5750854  -0.10845232  0.73735265 -0.95761815]
